# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For my label (`is_declining_label`, with the same definition as W04:
`trend_direction == 'down'`), I have a yes/no observed label, hence,
according to the training-honest-models skill, I would start with **Logistic
Regression** (interpretable – I can see what features increase/decrease the
score) and **Random Forest** (more powerful and still provides feature
importances). This time around, I will skip the Gradient Boosting model, as
the skill suggests that additional complexity should be added only after
justification from the comparison, and I'd like to see if the two simple
models already outperform my baseline before moving on to something more
complex.

As my lane's true output is a ranked list of items rather than yes/no, I
would evaluate the performance of the two models using **precision@K**
(K=20, K=50) based on predicted probabilities rather than simple
accuracy, since "ranking needs scores, not labels."

In [1]:
import pandas as pd, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    if not os.path.exists("internship"):
        get_ipython().system('git clone https://github.com/UnzilaAhsan/internship.git')
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

# Same prep as W01-W04: real visibility, not brand-new, one row per content item.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | base rate (declining): {df['is_declining_label'].mean():.3f}")


Cloning into 'internship'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 196 (delta 92), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 1.90 MiB | 4.86 MiB/s, done.
Resolving deltas: 100% (92/92), done.
30,000 rows | base rate (declining): 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-level holdout: 20% of clients are hold-out**, not 20% of rows. A random row split would make it possible to train the model on 80% of the pages of a certain client and then test it on 20% of the *same client's pages* — a walk in the park, as all the client-specific factors (the client's CMS, their niche, their CTR) were known in advance. A client-level holdout makes the right question clear: can it generalize to a completely unknown client? The exact split logic I've applied to check for data leakage in W03.

I am not doing any time-aware split — this feature vector is a 90-day snapshot for each piece of content, so there is no "future" period in it which could be hold-out separately from client identification.

In [2]:
# Client-aware split: hold out 20% of CLIENTS entirely (same logic as the W03 leakage check).
rng = np.random.default_rng(42)
clients = df["client_id"].unique()
shuffled = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = df["client_id"].isin(test_clients)

train_df = df[~test_mask].reset_index(drop=True)
test_df  = df[test_mask].reset_index(drop=True)

print(f"clients: {len(clients)}  |  held-out test clients: {len(test_clients)}")
print(f"train rows: {len(train_df):,}  (declining rate {train_df['is_declining_label'].mean():.3f})")
print(f"test rows:  {len(test_df):,}  (declining rate {test_df['is_declining_label'].mean():.3f})")
print("Note the declining rate itself differs between train and test -- that's expected with a "
      "client-grouped split (different clients, different mixes), and it's exactly the honest "
      "variation a random row split would have hidden.")


clients: 32  |  held-out test clients: 6
train rows: 27,675  (declining rate 0.555)
test rows:  2,325  (declining rate 0.391)
Note the declining rate itself differs between train and test -- that's expected with a client-grouped split (different clients, different mixes), and it's exactly the honest variation a random row split would have hidden.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Features:
same leakage-safe numeric+categorical combination as that in the reference pipeline of this repo
- raw and log 90-day traffic features, position, CTR, engagement, freshness, and content features.
`trend_direction`, `trend_pct` are not included (label is defined using them) along with
`impressions_prev_30d` and `impressions_last_30d` (too close to the trend to create circularity).

In [3]:
NUM = ["search_volume","competition","cpc","word_count","char_count",
       "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
       "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
       "word_count_tier","impression_tier","position_tier"]

def build_X(frame):
    num = frame[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).copy()
    num["log_impressions_90d"]  = np.log1p(frame["impressions_90d"].fillna(0))
    num["log_clicks_90d"]       = np.log1p(frame["clicks_90d"].fillna(0))
    num["log_sessions_90d"]     = np.log1p(frame["sessions_90d"].fillna(0))
    num["log_ai_sessions_90d"]  = np.log1p(frame["ai_sessions_90d"].fillna(0))
    cat = frame[CAT].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=CAT, dtype=float)
    return pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

X_train_raw, X_test_raw = build_X(train_df), build_X(test_df)
X_train, X_test = X_train_raw.align(X_test_raw, join="outer", axis=1, fill_value=0)
y_train, y_test = train_df["is_declining_label"], test_df["is_declining_label"]
print(f"feature count: {X_train.shape[1]}")

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    idx = np.argsort(-scores)[:k]
    return float(np.asarray(y_true)[idx].mean())

# --- Baseline: my W04 rule, recomputed on the TEST set only (same split, same data) ---
def pct_rank(s): return s.rank(pct=True)
tdf = test_df.copy()
tdf["visibility_score"] = pct_rank(np.log1p(tdf["impressions_90d"]))
tdf["position_gate"]    = ((tdf["avg_position"] > 0) & (tdf["avg_position"] <= 20)).astype(int)
tdf["low_ctr_score"]    = pct_rank(-tdf["ctr"])
baseline_scores = (tdf["visibility_score"] * tdf["position_gate"] * tdf["low_ctr_score"]).to_numpy()

results = {}
results["baseline (W04 rule)"] = {
    "precision@20": precision_at_k(y_test, baseline_scores, 20),
    "precision@50": precision_at_k(y_test, baseline_scores, 50),
    "roc_auc": roc_auc_score(y_test, baseline_scores),
    "avg_precision": average_precision_score(y_test, baseline_scores),
}

# --- Logistic Regression ---
lr = Pipeline([("scaler", StandardScaler()),
               ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]
results["logistic_regression"] = {
    "precision@20": precision_at_k(y_test, lr_scores, 20),
    "precision@50": precision_at_k(y_test, lr_scores, 50),
    "roc_auc": roc_auc_score(y_test, lr_scores),
    "avg_precision": average_precision_score(y_test, lr_scores),
}

# --- Random Forest ---
rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                             n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
results["random_forest"] = {
    "precision@20": precision_at_k(y_test, rf_scores, 20),
    "precision@50": precision_at_k(y_test, rf_scores, 50),
    "roc_auc": roc_auc_score(y_test, rf_scores),
    "avg_precision": average_precision_score(y_test, rf_scores),
}

print(f"\nbase rate on test set: {y_test.mean():.3f}\n")
comparison = pd.DataFrame(results).T.round(3)
print(comparison)


feature count: 52

base rate on test set: 0.391

                     precision@20  precision@50  roc_auc  avg_precision
baseline (W04 rule)          0.65          0.70    0.647          0.546
logistic_regression          0.35          0.40    0.700          0.522
random_forest                0.75          0.74    0.751          0.624


**Interpretation of the table**: The Random Forest outperforms the baseline model in all metrics (precision@20 = 0.75 against 0.65, precision@50 = 0.74 against 0.70, ROC AUC = 0.751 against 0.647). The interesting model here is the Logistic Regression: although it has a better ROC AUC than the baseline model (0.700 against 0.647), it has a lower precision@20 (0.35 against 0.65). This is not a paradox; this is the result of analysis: the Logistic Regression is good at ranking the entire population, but bad at ranking just the top portion, which is important for a review workflow. According to the Skill rule, both values should be considered, and therefore only Random Forest is a winner in this case.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# --- What the winning model (Random Forest) leans on ---
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 feature importances (Random Forest):")
print(importances.head(10).to_string())


Top 10 feature importances (Random Forest):
days_with_impressions    0.141586
log_impressions_90d      0.127133
avg_position             0.115721
content_age_days         0.096814
char_count               0.038103
word_count               0.037073
log_clicks_90d           0.035736
ctr                      0.035610
scroll_rate              0.033980
days_with_sessions       0.031531


**Sanity check on the key features:** `days_with_impressions`, `log_impressions_90d`,
`avg_position`, and `content_age_days` stand out - all reasonable, none suspiciously dominant (there
is no one feature that stands out above the rest as a leaked column would). Pages that have been
seen for more days, had more impressions overall, better ranking and age make perfect sense for a
refresh priority signal to rely on. There is nothing that can be linked to `trend_direction`, so this is
not the leaking signal from W03's intentional pitfall.

In [5]:
tdf["rf_prob"] = rf_scores
tdf["rf_pred"] = (tdf["rf_prob"] >= 0.5).astype(int)
wrong = tdf[tdf["rf_pred"] != tdf["is_declining_label"]]
print(f"Wrong on {len(wrong)} of {len(tdf)} test rows ({len(wrong)/len(tdf)*100:.1f}%)")

cols = ["content_id", "rf_prob", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "content_age_days"]

fp = wrong[wrong["is_declining_label"] == 0].sort_values("rf_prob", ascending=False).head(3)
print("\nTop 3 confident FALSE POSITIVES (predicted declining, actually stable):")
print(fp[cols].to_string(index=False))

fn = wrong[wrong["is_declining_label"] == 1].sort_values("rf_prob").head(3)
print("\nTop 3 confident FALSE NEGATIVES (predicted stable, actually declining):")
print(fn[cols].to_string(index=False))


Wrong on 756 of 2325 test rows (32.5%)

Top 3 confident FALSE POSITIVES (predicted declining, actually stable):
          content_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days
content_331182ca4cae 0.743898             3026          35.9  0.0                      20               134
content_e55b8ab078b0 0.739864              369          21.8  0.0                      20               112
content_db1cd41b4b4f 0.739258             1482          12.9  0.0                     105               105

Top 3 confident FALSE NEGATIVES (predicted stable, actually declining):
          content_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days
content_28b4223f4e5f 0.080485                1           0.0  0.0                       1                91
content_34b14c00f80c 0.098154                3           0.0  0.0                      20               308
content_79ac977c6e0b 0.133213                3           0.

**Why these are hard:** All three examples of false positives contain actual volume (369–3,026 impressions)
and zero CTR – they look just as they should look according to the model and baseline's understanding of what
"worth flagging" is but are labeled stable – and that is a plausible mistake rather than a faulty model.
False negatives are a case where the model ran out of information: 1-3 impressions, position virtually zero,
zero everything – there is literally nothing in these examples for a model to base its decisions on so it makes sense
to assume a low probability of decline regardless of the label being negative. Short story version: this model sacrifices a few uncertain low-CTR pages for
generalization across other clients and the gain in precision@20/50 is definitely worth it.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.